# SUCURSAL

In [0]:
catalog = spark.conf.get("bundle.catalog")
schema_bronze = spark.conf.get("bundle.schema_bronze")
landing_path = spark.conf.get("bundle.landing_path")

table_name = f"{catalog}.{schema_bronze}.sucursales"
source_path = f"{landing_path}/sucursales"

# 1. Crear la tabla con las 6 columnas en STRING
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_name} (
    sucursal_id STRING,
    nombre_sucursal STRING,
    distrito STRING,
    ciudad STRING,
    region STRING,
    fecha_apertura STRING
)
USING DELTA
COMMENT 'Tabla Bronze sucursales'
TBLPROPERTIES (
    'quality' = 'bronze',
    'pipelines.reset.allowed' = 'false',
    'delta.appendOnly' = 'true'
)
""")

# 2. Ejecutar COPY INTO manteniendo los tipos STRING
spark.sql(f"""
COPY INTO {table_name}
FROM (
    SELECT
        CAST(sucursal_id AS STRING) AS sucursal_id,
        CAST(nombre_sucursal AS STRING) AS nombre_sucursal,
        CAST(distrito AS STRING) AS distrito,
        CAST(ciudad AS STRING) AS ciudad,
        CAST(region AS STRING) AS region,
        CAST(fecha_apertura AS STRING) AS fecha_apertura
    FROM '{source_path}'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'delimiter' = ','
)
COPY_OPTIONS (
    'mergeSchema' = 'false'
)
""")